# TP - IRVE

Travail realise par :
- Issam Belhorma
- Zakaria Soual

In [41]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Partie 1 - Profiling & audit

- charger la copie locale du CSV consolide ;
- generer un profiling rapide dans le notebook ;
- auditer simplement le jeu de donnees sur 6 dimensions : completude, validite, coherence, unicite, exactitude et fraicheur ;
- mettre en lumiere les erreurs et approximations, sans corriger le dataset a ce stade.

### Chargement des donnees

On charge ici la copie locale du CSV fournie pour l'audit detaille, afin de travailler exactement sur le fichier reel analyse.

In [42]:
from pathlib import Path
import pandas as pd
from ydata_profiling import ProfileReport

CSV_PATH = Path('consolidation-etalab-schema-irve-statique-v-2.3.1-20260625.csv')

# low_memory=False evite des types trop instables pendant l'audit.
df = pd.read_csv(CSV_PATH, low_memory=False)
df_checkpoint = df.copy()

print(f'Fichier : {CSV_PATH.name}')
print(f"Nombre de lignes : {df.shape[0]:,}".replace(',', ' '))
print(f'Nombre de colonnes : {df.shape[1]}')

df.head()

Fichier : consolidation-etalab-schema-irve-statique-v-2.3.1-20260625.csv
Nombre de lignes : 227 232
Nombre de colonnes : 52


,nom_amenageur,siren_amenageur,contact_amenageur,nom_operateur,contact_operateur,telephone_operateur,nom_enseigne,id_station_itinerance,id_station_local,nom_station,...,datagouv_resource_id,datagouv_organization_or_owner,created_at,consolidated_longitude,consolidated_latitude,consolidated_code_postal,consolidated_commune,consolidated_is_lon_lat_correct,consolidated_is_code_insee_verified,consolidated_is_code_insee_modified
0,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004017,ATHTBE1004017,ACU_Poste_De_Garde_Haguenau,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.762694,48.825613,NaN,NaN,False,False,False
1,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004018,ATHTBE1004018,ACU_Poste_De_Garde_Haguenau,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.762694,48.825613,NaN,NaN,False,False,False
2,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004019,ATHTBE1004019,ACU_Poste_De_Garde_Haguenau,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.762694,48.825613,NaN,NaN,False,False,False
3,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004020,ATHTBE1004020,ACU_Poste_De_Garde_Haguenau,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.762694,48.825613,NaN,NaN,False,False,False
4,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,HAG_P22_Slave_3,ATHTBE1004957,ATHTBE1004957,HAG_P22_Slave_3,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.761841,48.827040,NaN,NaN,False,False,False


In [43]:
# Conversions utiles pour les controles.
date_mise_en_service_dt = pd.to_datetime(df['date_mise_en_service'], format='mixed', errors='coerce')
date_maj_dt = pd.to_datetime(df['date_maj'], format='mixed', errors='coerce')
puissance_nominale_num = pd.to_numeric(df['puissance_nominale'], errors='coerce')

# Completude sur toutes les colonnes du fichier.
completude = pd.DataFrame({
    'colonne': df.columns,
    'nb_manquants': df.isna().sum().values,
    'pct_manquants': (df.isna().mean() * 100).round(2).values,
}).sort_values('pct_manquants', ascending=False).reset_index(drop=True)

def lignes_manquantes(colonne, n=10):
    return df.loc[df[colonne].isna(), ['id_pdc_itinerance', 'nom_station', colonne]].head(n).copy()

completude_exemples = {
    colonne: lignes_manquantes(colonne)
    for colonne in completude.loc[completude['nb_manquants'] > 0, 'colonne']
}

### Rapport de profiling automatique

In [44]:
profile = ProfileReport(
    df,
    title='Partie 1 - Profiling initial IRVE',
    explorative=True,
    minimal=True,
)

profile.to_file('irve_profiling_report.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 52/52 [00:02<00:00, 22.42it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [45]:
# Creer les masques d'erreur : 
date_maj_regex = r'^\d{4}-\d{2}-\d{2}$'
last_modified_micro_regex = r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\.\d+\+00:00$'
telephone_regex = r'^\+?\d{9,15}$'
siren_regex = r'^\d{9}$'

siren_str = df['siren_amenageur'].astype('string').str.replace('.0', '', regex=False)
mask_siren = df['siren_amenageur'].notna() & ~siren_str.str.match(siren_regex)

telephone_norm = (
    df['telephone_operateur']
    .astype('string')
    .str.replace(r'(?i)^tel:', '', regex=True)
    .str.replace(r'[^0-9+]', '', regex=True)
)
mask_telephone = df['telephone_operateur'].notna() & ~telephone_norm.str.match(telephone_regex)

mask_puissance = df['puissance_nominale'].notna() & (
    puissance_nominale_num.isna()
    | (puissance_nominale_num <= 0)
    | (puissance_nominale_num > 1000)
)

mask_date_maj_format = df['date_maj'].notna() & ~df['date_maj'].astype('string').str.match(date_maj_regex)
mask_last_modified_format = df['last_modified'].notna() & ~df['last_modified'].astype('string').str.match(last_modified_micro_regex)

colonnes_booleennes = [
    'prise_type_ef', 'prise_type_2', 'prise_type_combo_ccs', 'prise_type_chademo',
    'prise_type_autre', 'gratuit', 'paiement_acte', 'paiement_cb',
    'reservation', 'station_deux_roues', 'cable_t2_attache',
]

booleens_long = (
    df[colonnes_booleennes]
    .reset_index()
    .rename(columns={'index': 'row_id'})
    .melt(id_vars='row_id', var_name='colonne', value_name='valeur')
)
mask_booleens = booleens_long['valeur'].notna() & ~booleens_long['valeur'].astype(str).str.lower().isin(['true', 'false'])
booleens_non_homogenes = (
    booleens_long.loc[mask_booleens]
    .merge(
        df[['id_pdc_itinerance', 'nom_station']].reset_index().rename(columns={'index': 'row_id'}),
        on='row_id',
        how='left',
    )
)

mask_coherence = date_mise_en_service_dt.notna() & date_maj_dt.notna() & (date_mise_en_service_dt > date_maj_dt)
mask_unicite = df['id_pdc_itinerance'].notna() & df['id_pdc_itinerance'].duplicated(keep=False)
mask_exactitude_coord = df['consolidated_is_lon_lat_correct'].astype('string').eq('False')
mask_exactitude_insee = df['consolidated_is_code_insee_verified'].astype('string').eq('False')
mask_fraicheur = date_maj_dt.notna() & (date_maj_dt < (date_maj_dt.max() - pd.Timedelta(days=365)))

resume_regles = []
erreurs_partie1 = {}

def ajoute_regle(dimension, regle, nb_erreurs, detail):
    resume_regles.append({
        'dimension': dimension,
        'regle': regle,
        'nb_erreurs': int(nb_erreurs),
        'pct_erreurs': round(int(nb_erreurs) / len(df) * 100, 2),
    })
    erreurs_partie1[f'{dimension} - {regle}'] = detail.head(20).copy()

ajoute_regle(
    'Validite',
    'SIREN amenageur non conforme',
    mask_siren.sum(),
    df.loc[mask_siren, ['siren_amenageur', 'nom_amenageur', 'nom_station']],
)
ajoute_regle(
    'Validite',
    'Telephone operateur non standard',
    mask_telephone.sum(),
    df.loc[mask_telephone, ['telephone_operateur', 'nom_operateur', 'nom_station']],
)
ajoute_regle(
    'Validite',
    'Puissance nominale invalide',
    mask_puissance.sum(),
    df.loc[mask_puissance, ['puissance_nominale', 'id_pdc_itinerance', 'nom_station']],
)
ajoute_regle(
    'Validite',
    'date_maj au mauvais format',
    mask_date_maj_format.sum(),
    df.loc[mask_date_maj_format, ['date_maj', 'id_pdc_itinerance', 'nom_station']],
)
ajoute_regle(
    'Validite',
    'last_modified non homogene',
    mask_last_modified_format.sum(),
    df.loc[mask_last_modified_format, ['last_modified', 'id_pdc_itinerance', 'nom_station']],
)
ajoute_regle(
    'Validite',
    'Booleens non homogenes',
    booleens_non_homogenes['row_id'].nunique(),
    booleens_non_homogenes[['colonne', 'valeur', 'id_pdc_itinerance', 'nom_station']],
)
ajoute_regle(
    'Coherence',
    'date_mise_en_service > date_maj',
    mask_coherence.sum(),
    df.loc[mask_coherence, ['date_mise_en_service', 'date_maj', 'id_pdc_itinerance', 'nom_station']],
)
ajoute_regle(
    'Unicite',
    'id_pdc_itinerance duplique',
    mask_unicite.sum(),
    df.loc[mask_unicite, ['id_pdc_itinerance', 'date_maj', 'last_modified', 'nom_station']],
)
ajoute_regle(
    'Exactitude',
    'Coordonnees signalees comme douteuses',
    mask_exactitude_coord.sum(),
    df.loc[mask_exactitude_coord, ['coordonneesXY', 'consolidated_longitude', 'consolidated_latitude', 'nom_station']],
)
ajoute_regle(
    'Exactitude',
    'Code INSEE non verifie',
    mask_exactitude_insee.sum(),
    df.loc[mask_exactitude_insee, ['code_insee_commune', 'consolidated_commune', 'consolidated_is_code_insee_verified', 'nom_station']],
)
ajoute_regle(
    'Fraicheur',
    'date_maj trop ancienne',
    mask_fraicheur.sum(),
    df.loc[mask_fraicheur, ['date_maj', 'id_pdc_itinerance', 'nom_station']],
)

audit_partie1 = pd.DataFrame(resume_regles).sort_values(['dimension', 'pct_erreurs'], ascending=[True, False]).reset_index(drop=True)
cles_erreurs = pd.DataFrame({'cle': list(erreurs_partie1.keys())})


### Audit simple des problemes principaux

On reste volontairement simple : on se limite a des regles lisibles, directement basees sur le contenu reel du CSV.

Aucune correction n'est faite dans cette partie : le but est uniquement de reperer et de montrer les anomalies.
Pour la completude, on garde une vue globale sur toutes les colonnes.
Pour les 5 autres dimensions, on extrait directement les lignes en erreur.

In [46]:
display(completude.head(15))

print("Exemple d'extraction des lignes manquantes :")
completude_exemples['consolidated_code_postal'].head()

,colonne,nb_manquants,pct_manquants
0,observations,180712,79.53
1,tarification,170887,75.20
2,num_pdl,111260,48.96
3,cable_t2_attache,106741,46.97
4,consolidated_code_postal,92870,40.87
5,date_mise_en_service,92271,40.61
6,siren_amenageur,90985,40.04
7,raccordement,79307,34.90
8,id_pdc_local,78678,34.62
9,consolidated_commune,76393,33.62


Exemple d'extraction des lignes manquantes :


,id_pdc_itinerance,nom_station,consolidated_code_postal
0,ATHTBE1004017,ACU_Poste_De_Garde_Haguenau,NaN
1,ATHTBE1004018,ACU_Poste_De_Garde_Haguenau,NaN
2,ATHTBE1004019,ACU_Poste_De_Garde_Haguenau,NaN
3,ATHTBE1004020,ACU_Poste_De_Garde_Haguenau,NaN
4,ATHTBE1004957,HAG_P22_Slave_3,NaN


In [47]:
display(audit_partie1)
display(cles_erreurs)

print("Exemples de requetes utiles :")
print("erreurs_partie1['Validite - date_maj au mauvais format'].head()")
print("erreurs_partie1['Coherence - date_mise_en_service > date_maj'].head()")
print("erreurs_partie1['Fraicheur - date_maj trop ancienne'].head()")

,dimension,regle,nb_erreurs,pct_erreurs
0,Coherence,date_mise_en_service > date_maj,2432,1.07
1,Exactitude,Coordonnees signalees comme douteuses,81724,35.97
2,Exactitude,Code INSEE non verifie,76393,33.62
3,Fraicheur,date_maj trop ancienne,80962,35.63
4,Unicite,id_pdc_itinerance duplique,130964,57.63
5,Validite,last_modified non homogene,33877,14.91
6,Validite,Booleens non homogenes,9926,4.37
7,Validite,Puissance nominale invalide,6257,2.75
8,Validite,Telephone operateur non standard,440,0.19
9,Validite,date_maj au mauvais format,293,0.13


,cle
0,Validite - SIREN amenageur non conforme
1,Validite - Telephone operateur non standard
2,Validite - Puissance nominale invalide
3,Validite - date_maj au mauvais format
4,Validite - last_modified non homogene
5,Validite - Booleens non homogenes
6,Coherence - date_mise_en_service > date_maj
7,Unicite - id_pdc_itinerance duplique
8,Exactitude - Coordonnees signalees comme doute...
9,Exactitude - Code INSEE non verifie


Exemples de requetes utiles :
erreurs_partie1['Validite - date_maj au mauvais format'].head()
erreurs_partie1['Coherence - date_mise_en_service > date_maj'].head()
erreurs_partie1['Fraicheur - date_maj trop ancienne'].head()


In [48]:
def score_dimension(series):
    if len(series) == 0:
        return 100.0
    return round(100 - series.mean(), 2)


scores = pd.DataFrame([
    {'dimension': 'Completude', 'score_initial': score_dimension(completude['pct_manquants'])},
    {'dimension': 'Validite', 'score_initial': score_dimension(audit_partie1.loc[audit_partie1['dimension'] == 'Validite', 'pct_erreurs'])},
    {'dimension': 'Coherence', 'score_initial': score_dimension(audit_partie1.loc[audit_partie1['dimension'] == 'Coherence', 'pct_erreurs'])},
    {'dimension': 'Unicite', 'score_initial': score_dimension(audit_partie1.loc[audit_partie1['dimension'] == 'Unicite', 'pct_erreurs'])},
    {'dimension': 'Exactitude', 'score_initial': score_dimension(audit_partie1.loc[audit_partie1['dimension'] == 'Exactitude', 'pct_erreurs'])},
    {'dimension': 'Fraicheur', 'score_initial': score_dimension(audit_partie1.loc[audit_partie1['dimension'] == 'Fraicheur', 'pct_erreurs'])},
])

score_global_initial = round(scores['score_initial'].mean(), 2)

display(scores)
print(f'Score de qualite global initial : {score_global_initial}/100')

,dimension,score_initial
0,Completude,87.73
1,Validite,96.27
2,Coherence,98.93
3,Unicite,42.37
4,Exactitude,65.20
5,Fraicheur,64.37


Score de qualite global initial : 75.81/100


## Partie 2 - Grain & doublons

In [49]:
df_partie2 = df_checkpoint.copy()

# Colonnes techniques pour ordonner les versions d'un meme point de charge.
df_partie2['date_maj_dt'] = pd.to_datetime(df_partie2['date_maj'], format='mixed', errors='coerce')
df_partie2['last_modified_dt'] = pd.to_datetime(df_partie2['last_modified'], format='mixed', errors='coerce')
df_partie2['created_at_dt'] = pd.to_datetime(df_partie2['created_at'], format='mixed', errors='coerce')

# Sous-ensemble des lignes touchees par le probleme d'unicite.
mask_doublons_pdc = df_partie2['id_pdc_itinerance'].notna() & df_partie2['id_pdc_itinerance'].duplicated(keep=False)
doublons_pdc = df_partie2.loc[mask_doublons_pdc].copy()

df_partie2.head()

,nom_amenageur,siren_amenageur,contact_amenageur,nom_operateur,contact_operateur,telephone_operateur,nom_enseigne,id_station_itinerance,id_station_local,nom_station,...,consolidated_longitude,consolidated_latitude,consolidated_code_postal,consolidated_commune,consolidated_is_lon_lat_correct,consolidated_is_code_insee_verified,consolidated_is_code_insee_modified,date_maj_dt,last_modified_dt,created_at_dt
0,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004017,ATHTBE1004017,ACU_Poste_De_Garde_Haguenau,...,7.762694,48.825613,NaN,NaN,False,False,False,2026-06-24,2026-06-24 21:00:21.100000+00:00,2023-06-28 11:46:08.539000+00:00
1,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004018,ATHTBE1004018,ACU_Poste_De_Garde_Haguenau,...,7.762694,48.825613,NaN,NaN,False,False,False,2026-06-24,2026-06-24 21:00:21.100000+00:00,2023-06-28 11:46:08.539000+00:00
2,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004019,ATHTBE1004019,ACU_Poste_De_Garde_Haguenau,...,7.762694,48.825613,NaN,NaN,False,False,False,2026-06-24,2026-06-24 21:00:21.100000+00:00,2023-06-28 11:46:08.539000+00:00
3,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004020,ATHTBE1004020,ACU_Poste_De_Garde_Haguenau,...,7.762694,48.825613,NaN,NaN,False,False,False,2026-06-24,2026-06-24 21:00:21.100000+00:00,2023-06-28 11:46:08.539000+00:00
4,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,HAG_P22_Slave_3,ATHTBE1004957,ATHTBE1004957,HAG_P22_Slave_3,...,7.761841,48.827040,NaN,NaN,False,False,False,2026-06-24,2026-06-24 21:00:21.100000+00:00,2023-06-28 11:46:08.539000+00:00


### Etude du grain

Une ligne correspond a une occurrence de point de charge dans une livraison. On mesure donc les repetitions de `id_pdc_itinerance`.

In [50]:
grain_resume = pd.DataFrame([
    {'indicateur': 'Nombre total de lignes', 'valeur': len(df_partie2)},
    {'indicateur': "Nombre d'id_pdc_itinerance non nuls", 'valeur': df_partie2['id_pdc_itinerance'].notna().sum()},
    {'indicateur': "Nombre d'id_pdc_itinerance uniques", 'valeur': df_partie2['id_pdc_itinerance'].nunique(dropna=True)},
    {'indicateur': 'Nombre de groupes dupliques', 'valeur': doublons_pdc['id_pdc_itinerance'].nunique()},
    {'indicateur': 'Nombre de lignes dans les groupes dupliques', 'valeur': len(doublons_pdc)},
    {'indicateur': "Nombre de lignes sans id_pdc_itinerance", 'valeur': df_partie2['id_pdc_itinerance'].isna().sum()},
])

grain_resume

,indicateur,valeur
0,Nombre total de lignes,227232
1,Nombre d'id_pdc_itinerance non nuls,227232
2,Nombre d'id_pdc_itinerance uniques,160138
3,Nombre de groupes dupliques,63870
4,Nombre de lignes dans les groupes dupliques,130964
5,Nombre de lignes sans id_pdc_itinerance,0


In [51]:
# Exemples de versions multiples pour un meme id.
exemples_repetes = (
    doublons_pdc
    .sort_values(['id_pdc_itinerance', 'date_maj_dt', 'last_modified_dt'], ascending=[True, True, True], na_position='last')
    [['id_pdc_itinerance', 'nom_station', 'date_maj', 'last_modified', 'datagouv_resource_id']]
    .head(20)
)

exemples_repetes

,id_pdc_itinerance,nom_station,date_maj,last_modified,datagouv_resource_id
27,ATHTBE1012062,ASIA AUTOMOTIVE,2021-10-13,2026-04-20T12:11:36.189000+00:00,5be52fb0-72ca-43c6-b88d-0d56fb3efff6
26,ATHTBE1012062,Asia Automotive,2026-06-24,2026-06-24T21:00:21.100000+00:00,b11113db-875d-41c7-8673-0cf8ad43e917
29,ATHTBE1012063,ASIA AUTOMOTIVE,2021-10-13,2026-04-20T12:11:36.189000+00:00,5be52fb0-72ca-43c6-b88d-0d56fb3efff6
28,ATHTBE1012063,Asia Automotive,2026-06-24,2026-06-24T21:00:21.100000+00:00,b11113db-875d-41c7-8673-0cf8ad43e917
895,FR190E190001SPE24ABB360121,Brive Watt'up Truck,2025-01-08,2025-01-17T06:30:12.104000+00:00,cd8bcc95-82ec-461b-979e-a07f2f14f4a0
894,FR190E190001SPE24ABB360121,Watt up by Enerjump - Brive 19-01,2026-06-24,2026-06-24T21:00:21.100000+00:00,b11113db-875d-41c7-8673-0cf8ad43e917
897,FR190E190001SPE24ABB360122,Brive Watt'up Truck,2025-01-08,2025-01-17T06:30:12.104000+00:00,cd8bcc95-82ec-461b-979e-a07f2f14f4a0
896,FR190E190001SPE24ABB360122,Watt up by Enerjump - Brive 19-01,2026-06-24,2026-06-24T21:00:21.100000+00:00,b11113db-875d-41c7-8673-0cf8ad43e917
899,FR190E190001SPE24ABB360341,Brive Watt'up Truck,2025-01-08,2025-01-17T06:30:12.104000+00:00,cd8bcc95-82ec-461b-979e-a07f2f14f4a0
898,FR190E190001SPE24ABB360341,Watt up by Enerjump - Brive 19-01,2026-06-24,2026-06-24T21:00:21.100000+00:00,b11113db-875d-41c7-8673-0cf8ad43e917


### Regle de dedoublonnage retenue

Cle de dedoublonnage : `id_pdc_itinerance`.

On garde une seule ligne par identifiant.

Ordre de priorite :
- `last_modified_dt` decroissante ;
- `date_maj_dt` decroissante ;
- `created_at_dt` decroissante.

Les lignes sans `id_pdc_itinerance` sont conservees telles quelles.

In [52]:
# Dedoublonnage uniquement sur les lignes avec identifiant.
avec_id = df_partie2[df_partie2['id_pdc_itinerance'].notna()].copy()
sans_id = df_partie2[df_partie2['id_pdc_itinerance'].isna()].copy()

avec_id_dedup = (
    avec_id
    .sort_values(
        ['id_pdc_itinerance', 'last_modified_dt', 'date_maj_dt', 'created_at_dt'],
        ascending=[True, False, False, False],
        na_position='last',
    )
    .drop_duplicates(subset='id_pdc_itinerance', keep='first')
)

df_dedoublonne = pd.concat([avec_id_dedup, sans_id], ignore_index=True)

colonnes_techniques = ['last_modified_dt', 'date_maj_dt', 'created_at_dt']
df_dedoublonne = df_dedoublonne.drop(columns=colonnes_techniques)

df_dedoublonne.head()

,nom_amenageur,siren_amenageur,contact_amenageur,nom_operateur,contact_operateur,telephone_operateur,nom_enseigne,id_station_itinerance,id_station_local,nom_station,...,datagouv_resource_id,datagouv_organization_or_owner,created_at,consolidated_longitude,consolidated_latitude,consolidated_code_postal,consolidated_commune,consolidated_is_lon_lat_correct,consolidated_is_code_insee_verified,consolidated_is_code_insee_modified
0,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004017,ATHTBE1004017,ACU_Poste_De_Garde_Haguenau,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.762694,48.825613,NaN,NaN,False,False,False
1,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004018,ATHTBE1004018,ACU_Poste_De_Garde_Haguenau,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.762694,48.825613,NaN,NaN,False,False,False
2,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004019,ATHTBE1004019,ACU_Poste_De_Garde_Haguenau,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.762694,48.825613,NaN,NaN,False,False,False
3,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,ACU_Poste_De_Garde_Haguenau,ATHTBE1004020,ATHTBE1004020,ACU_Poste_De_Garde_Haguenau,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.762694,48.825613,NaN,NaN,False,False,False
4,ChargePoint,NaN,info@chargepoint.com,ChargePoint,info@chargepoint.com,NaN,HAG_P22_Slave_3,ATHTBE1004957,ATHTBE1004957,HAG_P22_Slave_3,...,b11113db-875d-41c7-8673-0cf8ad43e917,eco-movement,2023-06-28T11:46:08.539000+00:00,7.761841,48.827040,NaN,NaN,False,False,False


### Impact du dedoublonnage

On compare les volumes avant / apres et on verifie qu'il ne reste plus de doublons sur `id_pdc_itinerance`.

In [53]:
restants_doublons = (
    df_dedoublonne['id_pdc_itinerance'].notna()
    & df_dedoublonne['id_pdc_itinerance'].duplicated(keep=False)
)

impact_dedoublonnage = pd.DataFrame([
    {'indicateur': 'Nombre de lignes avant dedoublonnage', 'valeur': len(df_partie2)},
    {'indicateur': 'Nombre de lignes apres dedoublonnage', 'valeur': len(df_dedoublonne)},
    {'indicateur': 'Nombre de lignes retirees', 'valeur': len(df_partie2) - len(df_dedoublonne)},
    {'indicateur': "Nombre d'id_pdc_itinerance uniques avant", 'valeur': df_partie2['id_pdc_itinerance'].nunique(dropna=True)},
    {'indicateur': "Nombre d'id_pdc_itinerance uniques apres", 'valeur': df_dedoublonne['id_pdc_itinerance'].nunique(dropna=True)},
    {'indicateur': 'Nombre de lignes encore dupliquees apres', 'valeur': int(restants_doublons.sum())},
])

impact_dedoublonnage

,indicateur,valeur
0,Nombre de lignes avant dedoublonnage,227232
1,Nombre de lignes apres dedoublonnage,160138
2,Nombre de lignes retirees,67094
3,Nombre d'id_pdc_itinerance uniques avant,160138
4,Nombre d'id_pdc_itinerance uniques apres,160138
5,Nombre de lignes encore dupliquees apres,0


## Partie 3 - Nettoyage & enrichissement

On repart du dataset dedoublonne et on applique des corrections simples sur les anomalies mises en evidence dans les parties 1 et 2.

In [ ]:
import re

df_partie3_avant = df_dedoublonne.copy()
df_partie3 = df_dedoublonne.copy()

colonnes_texte = df_partie3.select_dtypes(include='object').columns.tolist()
colonnes_booleennes = [
    'prise_type_ef', 'prise_type_2', 'prise_type_combo_ccs', 'prise_type_chademo',
    'prise_type_autre', 'gratuit', 'paiement_acte', 'paiement_cb',
    'reservation', 'station_deux_roues', 'cable_t2_attache',
]

motif_encodage = r'č|Ã|Â|¤|�|â¬|Ã¯Â¿Â½'

def nb_cellules_suspectes(frame):
    total = 0
    for col in frame.select_dtypes(include='object').columns:
        total += frame[col].astype('string').str.contains(motif_encodage, regex=True, na=False).sum()
    return int(total)

def nb_lignes_booleennes_non_homogenes(frame):
    booleens_long = (
        frame[colonnes_booleennes]
        .reset_index()
        .rename(columns={'index': 'row_id'})
        .melt(id_vars='row_id', var_name='colonne', value_name='valeur')
    )
    mask = booleens_long['valeur'].notna() & ~booleens_long['valeur'].astype(str).str.lower().isin(['true', 'false'])
    return int(booleens_long.loc[mask, 'row_id'].nunique())

df_partie3.head()

### Encodage

On corrige les libelles corrompus sur les colonnes texte.

In [ ]:
suspects_encodage_avant = nb_cellules_suspectes(df_partie3)

corrections_texte = {
    'Accčs': 'Accès',
    'Extčrieur': 'Extérieur',
    'Intčrieur': 'Intérieur',
    'lčs': 'lès',
    'Noyčres': 'Noyères',
    'Glaci�res': 'Glacières',
    'Pin�de': 'Pinède',
    'L�on': 'Léon',
    'Ferričre': 'Ferrière',
    'MÃĐrignac': 'Mérignac',
    'prÃ©cisÃ©e': 'précisée',
    'mÃ¨tres': 'mètres',
    'opÃ¯Â¿Â½rateur': 'opérateur',
    'mobilitÃ¯Â¿Â½': 'mobilité',
    'Ã¯Â¿Â½lectrique': 'électrique',
    'op�rateur': 'opérateur',
    'mobilit�': 'mobilité',
    '�lectrique': 'électrique',
    'rÃ©seau': 'réseau',
    'RÃ©seau': 'Réseau',
    'rÃ©publique': 'république',
    'dÃ©faut': 'défaut',
    'prÃ©alable': 'préalable',
    'centaureÃÅes': 'centaurées',
    'ArdÃÂ¨che': 'Ardèche',
    'RhÃÂ´ne': 'Rhône',
}

def repare_texte(value):
    if pd.isna(value):
        return value
    text = str(value)
    for mauvais, bon in corrections_texte.items():
        text = text.replace(mauvais, bon)
    text = text.replace('č', 'è').replace('Č', 'È').replace('â¬', '€')
    if 'Ã' in text or 'Â' in text:
        try:
            text = text.encode('latin1').decode('utf-8')
        except Exception:
            pass
    text = text.replace('Â', '')
    text = re.sub(r'\\s+', ' ', text).strip()
    return text

for col in colonnes_texte:
    df_partie3[col] = df_partie3[col].map(repare_texte)

suspects_encodage_apres = nb_cellules_suspectes(df_partie3)
encodage_resume = pd.DataFrame([
    {'indicateur': 'Cellules suspectes avant', 'valeur': suspects_encodage_avant},
    {'indicateur': 'Cellules suspectes apres', 'valeur': suspects_encodage_apres},
    {'indicateur': 'Cellules corrigees', 'valeur': suspects_encodage_avant - suspects_encodage_apres},
])

display(encodage_resume)
df_partie3.loc[df_partie3['condition_acces'].astype('string').str.contains('Acces', na=False), ['condition_acces']].head(10)

### Formats

On homogeneise les booleens, les communes et les dates.

In [ ]:
booleens_non_homogenes_avant = nb_lignes_booleennes_non_homogenes(df_partie3)

mapping_booleens = {
    '1': 'true', '0': 'false',
    'TRUE': 'true', 'FALSE': 'false',
    'True': 'true', 'False': 'false',
    'true': 'true', 'false': 'false',
}

for col in colonnes_booleennes:
    df_partie3[col] = df_partie3[col].map(lambda x: mapping_booleens.get(str(x), x) if pd.notna(x) else x)

communes_avant = df_partie3['consolidated_commune'].copy()

def normalise_commune(value):
    if pd.isna(value):
        return value
    text = re.sub(r'\\s+', ' ', str(value)).strip()
    if text.isupper() or text.islower():
        text = text.title()
        text = text.replace("-D'", "-d'").replace(" D'", " d'")
    return text

df_partie3['consolidated_commune'] = df_partie3['consolidated_commune'].map(normalise_commune)
communes_modifiees = int((communes_avant.fillna('') != df_partie3['consolidated_commune'].fillna('')).sum())

date_maj_non_standard_avant = int((df_partie3['date_maj'].dropna().astype(str).str.match(r'^\\d{4}-\\d{2}-\\d{2}$') == False).sum())

for col in ['date_mise_en_service', 'date_maj']:
    dt = pd.to_datetime(df_partie3[col], format='mixed', errors='coerce')
    df_partie3[col] = dt.dt.strftime('%Y-%m-%d').where(dt.notna(), pd.NA)

for col in ['last_modified', 'created_at']:
    dt = pd.to_datetime(df_partie3[col], format='mixed', utc=True, errors='coerce')
    df_partie3[col] = dt.dt.strftime('%Y-%m-%dT%H:%M:%S.%f+00:00').where(dt.notna(), pd.NA)

date_maj_non_standard_apres = int((df_partie3['date_maj'].dropna().astype(str).str.match(r'^\\d{4}-\\d{2}-\\d{2}$') == False).sum())
booleens_non_homogenes_apres = nb_lignes_booleennes_non_homogenes(df_partie3)

formats_resume = pd.DataFrame([
    {'indicateur': 'Lignes booleennes non homogenes avant', 'valeur': booleens_non_homogenes_avant},
    {'indicateur': 'Lignes booleennes non homogenes apres', 'valeur': booleens_non_homogenes_apres},
    {'indicateur': 'Communes modifiees', 'valeur': communes_modifiees},
    {'indicateur': 'date_maj hors format avant', 'valeur': date_maj_non_standard_avant},
    {'indicateur': 'date_maj hors format apres', 'valeur': date_maj_non_standard_apres},
])

formats_resume

### Codes Postaux

On corrige les codes postaux invalides par cascade : adresse identique, coordonnees exactes, puis ville la plus proche.

In [ ]:
import numpy as np
from scipy.spatial import cKDTree

def normalise_adresse(value):
    if pd.isna(value):
        return pd.NA
    text = re.sub(r'\s+', ' ', str(value)).strip().lower()
    return text if text else pd.NA

cp_avant = pd.to_numeric(df_partie3['consolidated_code_postal'], errors='coerce')
mask_cp_invalide = cp_avant.isna() | (cp_avant < 1000) | (cp_avant > 99999)

df_partie3['adresse_norm'] = df_partie3['adresse_station'].map(normalise_adresse)
df_partie3['cp_travail'] = cp_avant
df_partie3['source_cp'] = pd.NA

reference_cp = df_partie3.loc[~mask_cp_invalide].copy()

# 1. Meme adresse -> meme code postal.
reference_adresse = (
    reference_cp[reference_cp['adresse_norm'].notna()]
    .groupby('adresse_norm')['cp_travail']
    .agg(lambda s: s.dropna().mode().iloc[0] if not s.dropna().mode().empty else pd.NA)
    .reset_index(name='cp_par_adresse')
)
df_partie3 = df_partie3.merge(reference_adresse, on='adresse_norm', how='left')
mask_fill_adresse = mask_cp_invalide & df_partie3['cp_par_adresse'].notna()
df_partie3.loc[mask_fill_adresse, 'cp_travail'] = df_partie3.loc[mask_fill_adresse, 'cp_par_adresse']
df_partie3.loc[mask_fill_adresse, 'source_cp'] = 'adresse'

# 2. Memes coordonnees -> meme code postal.
mask_cp_restant = df_partie3['cp_travail'].isna() | (df_partie3['cp_travail'] < 1000) | (df_partie3['cp_travail'] > 99999)
reference_coordonnees = (
    reference_cp[
        reference_cp['consolidated_longitude'].notna()
        & reference_cp['consolidated_latitude'].notna()
    ]
    .groupby(['consolidated_longitude', 'consolidated_latitude'])['cp_travail']
    .agg(lambda s: s.dropna().mode().iloc[0] if not s.dropna().mode().empty else pd.NA)
    .reset_index(name='cp_par_coordonnees')
)
df_partie3 = df_partie3.merge(reference_coordonnees, on=['consolidated_longitude', 'consolidated_latitude'], how='left')
mask_fill_coordonnees = mask_cp_restant & df_partie3['cp_par_coordonnees'].notna()
df_partie3.loc[mask_fill_coordonnees, 'cp_travail'] = df_partie3.loc[mask_fill_coordonnees, 'cp_par_coordonnees']
df_partie3.loc[mask_fill_coordonnees, 'source_cp'] = 'coordonnees_exactes'

# 3. Coordonnees restantes -> ville la plus proche.
mask_cp_restant = df_partie3['cp_travail'].isna() | (df_partie3['cp_travail'] < 1000) | (df_partie3['cp_travail'] > 99999)
reference_villes = (
    reference_cp[
        reference_cp['consolidated_commune'].notna()
        & reference_cp['consolidated_longitude'].notna()
        & reference_cp['consolidated_latitude'].notna()
    ]
    .groupby('consolidated_commune')
    .agg(
        longitude_ref=('consolidated_longitude', 'median'),
        latitude_ref=('consolidated_latitude', 'median'),
        cp_ville=('cp_travail', lambda s: s.dropna().mode().iloc[0] if not s.dropna().mode().empty else pd.NA),
    )
    .reset_index()
)

mask_ville_proche = mask_cp_restant & df_partie3['consolidated_longitude'].notna() & df_partie3['consolidated_latitude'].notna()

if mask_ville_proche.any() and not reference_villes.empty:
    tree = cKDTree(reference_villes[['longitude_ref', 'latitude_ref']].to_numpy())
    _, index_plus_proche = tree.query(
        df_partie3.loc[mask_ville_proche, ['consolidated_longitude', 'consolidated_latitude']].to_numpy(),
        k=1,
    )
    ville_plus_proche = reference_villes.iloc[index_plus_proche].reset_index(drop=True)

    lon1 = np.radians(df_partie3.loc[mask_ville_proche, 'consolidated_longitude'].to_numpy())
    lat1 = np.radians(df_partie3.loc[mask_ville_proche, 'consolidated_latitude'].to_numpy())
    lon2 = np.radians(ville_plus_proche['longitude_ref'].to_numpy())
    lat2 = np.radians(ville_plus_proche['latitude_ref'].to_numpy())
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    distance_km = 2 * 6371 * np.arcsin(np.sqrt(a))

    df_partie3.loc[mask_ville_proche, 'commune_proche'] = ville_plus_proche['consolidated_commune'].to_numpy()
    df_partie3.loc[mask_ville_proche, 'cp_ville_proche'] = ville_plus_proche['cp_ville'].to_numpy()
    df_partie3.loc[mask_ville_proche, 'distance_ville_km'] = distance_km

    mask_fill_ville = mask_ville_proche & df_partie3['cp_ville_proche'].notna()
    df_partie3.loc[mask_fill_ville, 'cp_travail'] = df_partie3.loc[mask_fill_ville, 'cp_ville_proche']
    df_partie3.loc[mask_fill_ville, 'source_cp'] = 'ville_plus_proche'
    df_partie3.loc[mask_fill_ville & df_partie3['consolidated_commune'].isna(), 'consolidated_commune'] = df_partie3.loc[mask_fill_ville & df_partie3['consolidated_commune'].isna(), 'commune_proche']
else:
    mask_fill_ville = pd.Series(False, index=df_partie3.index)

exemples_cp_imputes = df_partie3.loc[
    mask_fill_adresse | mask_fill_coordonnees | mask_fill_ville,
    [
        'adresse_station', 'consolidated_longitude', 'consolidated_latitude',
        'consolidated_commune', 'cp_travail', 'source_cp', 'nom_station',
    ],
].head(20).copy()

df_partie3['consolidated_code_postal'] = df_partie3['cp_travail']
mask_cp_restant = df_partie3['consolidated_code_postal'].isna() | (df_partie3['consolidated_code_postal'] < 1000) | (df_partie3['consolidated_code_postal'] > 99999)

cp_resume = pd.DataFrame([
    {'indicateur': 'Codes postaux invalides avant', 'valeur': int(mask_cp_invalide.sum())},
    {'indicateur': 'Corrections par adresse', 'valeur': int(mask_fill_adresse.sum())},
    {'indicateur': 'Corrections par coordonnees exactes', 'valeur': int(mask_fill_coordonnees.sum())},
    {'indicateur': 'Corrections par ville la plus proche', 'valeur': int(mask_fill_ville.sum())},
    {'indicateur': 'Codes postaux invalides apres', 'valeur': int(mask_cp_restant.sum())},
])

df_partie3 = df_partie3.drop(columns=[
    'adresse_norm', 'cp_travail', 'source_cp', 'cp_par_adresse', 'cp_par_coordonnees',
    'commune_proche', 'cp_ville_proche', 'distance_ville_km',
], errors='ignore')

display(cp_resume)
exemples_cp_imputes

### Puissance

On interprete les valeurs > 1000 comme des W saisis a la place des kW, puis on impute les autres valeurs aberrantes par la mediane.

In [ ]:
import numpy as np

puissance_brute = pd.to_numeric(df_partie3['puissance_nominale'], errors='coerce')

# Valeurs > 1000 : on les traite comme des W et on les remet en kW.
mask_puissance_w = puissance_brute > 1000
puissance_kw = puissance_brute.copy()
puissance_kw.loc[mask_puissance_w] = puissance_kw.loc[mask_puissance_w] / 1000

# Valeurs aberrantes restantes : <= 0 ou non numeriques.
mask_puissance_non_positive = puissance_kw <= 0
mask_puissance_non_numerique = df_partie3['puissance_nominale'].notna() & puissance_kw.isna()
mask_puissance_a_imputer = mask_puissance_non_positive | mask_puissance_non_numerique

mediane_puissance_kw = float(puissance_kw[(puissance_kw > 0) & puissance_kw.notna()].median())

index_exemples = df_partie3.index[mask_puissance_w | mask_puissance_a_imputer][:20]
exemples_puissance = df_partie3.loc[index_exemples, ['id_pdc_itinerance', 'nom_station']].copy()
exemples_puissance['puissance_avant'] = puissance_brute.loc[index_exemples].values
exemples_puissance['action'] = np.where(mask_puissance_w.loc[index_exemples], 'W_vers_kW', 'mediane')

df_partie3.loc[mask_puissance_w, 'puissance_nominale'] = puissance_kw.loc[mask_puissance_w]
df_partie3.loc[mask_puissance_a_imputer, 'puissance_nominale'] = mediane_puissance_kw

puissance_apres = pd.to_numeric(df_partie3['puissance_nominale'], errors='coerce')
exemples_puissance['puissance_apres'] = puissance_apres.loc[index_exemples].values

puissance_resume = pd.DataFrame([
    {'indicateur': 'Valeurs interpretees comme W puis converties en kW', 'valeur': int(mask_puissance_w.sum())},
    {'indicateur': 'Valeurs imputees par la mediane', 'valeur': int(mask_puissance_a_imputer.sum())},
    {'indicateur': 'Mediane utilisee (kW)', 'valeur': round(mediane_puissance_kw, 2)},
    {'indicateur': 'Valeurs encore invalides apres traitement', 'valeur': int(((puissance_apres <= 0) | (puissance_apres > 1000) | puissance_apres.isna()).sum())},
])

display(puissance_resume)
exemples_puissance

### Score Apres Nettoyage

On recalcule un audit simple sur le dataset nettoye pour mesurer l'effet des corrections.

In [ ]:
def calcule_audit_simple(frame):
    completude_frame = pd.DataFrame({
        'colonne': frame.columns,
        'pct_manquants': (frame.isna().mean() * 100).round(2).values,
    })

    date_mise_en_service_dt = pd.to_datetime(frame['date_mise_en_service'], format='mixed', errors='coerce')
    date_maj_dt = pd.to_datetime(frame['date_maj'], format='mixed', errors='coerce')
    puissance_num = pd.to_numeric(frame['puissance_nominale'], errors='coerce')

    siren_str = frame['siren_amenageur'].astype('string').str.replace('.0', '', regex=False)
    telephone_norm = (
        frame['telephone_operateur']
        .astype('string')
        .str.replace(r'(?i)^tel:', '', regex=True)
        .str.replace(r'[^0-9+]', '', regex=True)
    )

    booleens_long = (
        frame[colonnes_booleennes]
        .reset_index()
        .rename(columns={'index': 'row_id'})
        .melt(id_vars='row_id', var_name='colonne', value_name='valeur')
    )
    mask_booleens = booleens_long['valeur'].notna() & ~booleens_long['valeur'].astype(str).str.lower().isin(['true', 'false'])

    audit = pd.DataFrame([
        {'dimension': 'Validite', 'regle': 'SIREN amenageur non conforme', 'pct_erreurs': round((frame['siren_amenageur'].notna() & ~siren_str.str.match(r'^\\d{9}$')).mean() * 100, 2)},
        {'dimension': 'Validite', 'regle': 'Telephone operateur non standard', 'pct_erreurs': round((frame['telephone_operateur'].notna() & ~telephone_norm.str.match(r'^\\+?\\d{9,15}$')).mean() * 100, 2)},
        {'dimension': 'Validite', 'regle': 'Puissance nominale invalide', 'pct_erreurs': round((frame['puissance_nominale'].notna() & ((puissance_num <= 0) | (puissance_num > 1000) | puissance_num.isna())).mean() * 100, 2)},
        {'dimension': 'Validite', 'regle': 'date_maj au mauvais format', 'pct_erreurs': round((frame['date_maj'].notna() & ~frame['date_maj'].astype('string').str.match(r'^\\d{4}-\\d{2}-\\d{2}$')).mean() * 100, 2)},
        {'dimension': 'Validite', 'regle': 'Booleens non homogenes', 'pct_erreurs': round(mask_booleens.mean() * 100, 2)},
        {'dimension': 'Coherence', 'regle': 'date_mise_en_service > date_maj', 'pct_erreurs': round((date_mise_en_service_dt.notna() & date_maj_dt.notna() & (date_mise_en_service_dt > date_maj_dt)).mean() * 100, 2)},
        {'dimension': 'Unicite', 'regle': 'id_pdc_itinerance duplique', 'pct_erreurs': round((frame['id_pdc_itinerance'].notna() & frame['id_pdc_itinerance'].duplicated(keep=False)).mean() * 100, 2)},
        {'dimension': 'Exactitude', 'regle': 'Coordonnees signalees comme douteuses', 'pct_erreurs': round(frame['consolidated_is_lon_lat_correct'].astype('string').eq('False').mean() * 100, 2)},
        {'dimension': 'Exactitude', 'regle': 'Code INSEE non verifie', 'pct_erreurs': round(frame['consolidated_is_code_insee_verified'].astype('string').eq('False').mean() * 100, 2)},
        {'dimension': 'Fraicheur', 'regle': 'date_maj trop ancienne', 'pct_erreurs': round((date_maj_dt.notna() & (date_maj_dt < (date_maj_dt.max() - pd.Timedelta(days=365)))).mean() * 100, 2)},
    ])

    scores = pd.DataFrame([
        {'dimension': 'Completude', 'score': score_dimension(completude_frame['pct_manquants'])},
        {'dimension': 'Validite', 'score': score_dimension(audit.loc[audit['dimension'] == 'Validite', 'pct_erreurs'])},
        {'dimension': 'Coherence', 'score': score_dimension(audit.loc[audit['dimension'] == 'Coherence', 'pct_erreurs'])},
        {'dimension': 'Unicite', 'score': score_dimension(audit.loc[audit['dimension'] == 'Unicite', 'pct_erreurs'])},
        {'dimension': 'Exactitude', 'score': score_dimension(audit.loc[audit['dimension'] == 'Exactitude', 'pct_erreurs'])},
        {'dimension': 'Fraicheur', 'score': score_dimension(audit.loc[audit['dimension'] == 'Fraicheur', 'pct_erreurs'])},
    ])

    score_global = round(scores['score'].mean(), 2)
    return audit, scores, score_global

audit_partie3_avant, scores_partie3_avant, score_global_partie3_avant = calcule_audit_simple(df_partie3_avant)
audit_partie3_apres, scores_partie3_apres, score_global_partie3_apres = calcule_audit_simple(df_partie3)

comparaison_scores_partie3 = scores_partie3_avant.merge(
    scores_partie3_apres,
    on='dimension',
    suffixes=('_avant', '_apres'),
)

df_nettoye = df_partie3.copy()

display(comparaison_scores_partie3)
display(audit_partie3_apres)
print(f'Score global avant nettoyage : {score_global_partie3_avant}/100')
print(f'Score global apres nettoyage : {score_global_partie3_apres}/100')